In [ ]:
# Check Modal is installed.
!uv run modal --version

### Simple examples

In [ ]:
# Run code remotely on Modal from "shell".
!uv run modal run modal_hello_world.py

In [ ]:
# Run code remotely on Modal from notebook.
import modal

app = modal.App("example-get-started")
# container = app.run()

@app.function()
def square(x):
    print("This code is running on a remote worker!")
    return x**2

with app.run():
    print("the square is", square.remote(42))

### Run remotely my LLM

In [ ]:
import modal

# Create an app will all needed packages.
app_image = (
    modal.Image.debian_slim()
    # Install cs336-basics before pip_install_from_pyproject tries (and fails)
    # to install it from PyPI.
    .apt_install("git")
    .run_commands(
        "git clone https://github.com/NiccoloSacchi/assignment1-basics /root/assignment1-basics",
        "pip install -e /root/assignment1-basics",
    )
    # This installs only the dependencies list, ignoring the rest, i.e. it does
    # not know that cs336-basics should be installed from a local path.
    .pip_install_from_pyproject("/Users/niccolosacchi/assignment2-systems/pyproject.toml")
)
app = modal.App("cs336-systems", image=app_image)

In [ ]:
# Define the remote function with a GPU.
@app.function(gpu="any")  # Any GPU is fine for faster scheduling.
def run_transformer():
    from cs336_basics.model import TransformerLM
    import torch
    
    # Use CUDA instead of MPS
    device = torch.device("cuda")
    
    model = TransformerLM(
        vocab_size=10000,
        context_length=256,
        num_layers=8,
        d_model=768,
        num_heads=16,
        d_ff=1344,
        rope_theta=10000.0,
        device=device,
        dtype=torch.float32,
    )

    input_tokens = torch.randint(0, 10000, (1, 256), device=device)
    output = model(input_tokens)
    
    print(f"Success! Output shape: {output.shape}")
    return output.cpu()

with app.run():
    output = run_transformer.remote()
    
output